# SWT Fraud Detection Pipeline
Step-by-step execution with column names printed after every step.

**Run this notebook from inside the `swt/` folder.**

The SWT pipeline runs each script as a subprocess. This notebook runs each script
individually, then reads the saved intermediate file to print column names.

In [ ]:
import os, sys, subprocess, tempfile, time, warnings
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# ── SET THIS to your absolute path of the swt/ folder ──────────────────────
SWT_DIR = os.path.dirname(os.path.abspath('swt_fraud_pipeline_with_timer.py'))
os.chdir(SWT_DIR)
sys.path.insert(0, SWT_DIR)
sys.path.insert(0, os.path.dirname(SWT_DIR))   # project root

OUTPUT_DIR = Path(SWT_DIR) / 'final_output'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Working directory : {os.getcwd()}')
print(f'Output directory  : {OUTPUT_DIR}')

In [ ]:
# Helper: run a python script and stream its output
def run_script(script_path):
    env = dict(os.environ)
    env['SWT_OUTPUT_DIR'] = str(OUTPUT_DIR.resolve())
    env['SWT_MODELS_DIR'] = str(Path(SWT_DIR) / 'models')
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=SWT_DIR,
        env=env
    )
    return result.returncode == 0

# Helper: find the SWT input file in data/
data_dir = Path(SWT_DIR) / 'Data'
all_files = list(data_dir.glob('*.parquet')) + list(data_dir.glob('*.csv'))
swt_files = [f for f in all_files if 'swt' in f.name.lower()]
candidates = swt_files if swt_files else all_files
input_file = max(candidates, key=lambda f: f.stat().st_mtime) if candidates else None
print(f'Input file: {input_file}')

## RAW INPUT — Columns before any processing

In [ ]:
if input_file.suffix == '.parquet':
    raw_df = pd.read_parquet(input_file)
else:
    raw_df = pd.read_csv(input_file)

print(f'Raw data shape: {raw_df.shape}')
print(f'\n=== RAW INPUT COLUMNS ({len(raw_df.columns)}) ===')
print(raw_df.columns.tolist())

## STEP 1 — Data Preparation & Standardization
Script: `1_swt_preparation.py`  →  saves `final_output/swt_standardized.parquet`

In [ ]:
script1 = Path(SWT_DIR) / '1_swt_preparation.py'
print(f'Running: {script1.name} ...')
ok = run_script(script1)
print(f'Step 1 finished, success={ok}')

# ── PRINT COLUMNS AFTER STEP 1 ───────────────────────────────────────────────
out1 = OUTPUT_DIR / 'swt_standardized.parquet'
df_step1 = pd.read_parquet(out1)
print(f'\n=== COLUMNS AFTER STEP 1 — Standardization ({len(df_step1.columns)}) ===')
print(df_step1.columns.tolist())

## STEP 2 — Data Validation & Cleaning
Script: `2_swt_validation.py`  →  saves `swt_data_after_taxpayer_name_mapping.parquet`

In [ ]:
# Patch the validation script to auto-answer interactive prompts
script2_orig = Path(SWT_DIR) / '2_swt_validation.py'
with open(script2_orig, 'r', encoding='utf-8') as f:
    content = f.read()

content = content.replace(
    'response = input("Do you want to proceed with these records? (yes/no): ").strip().lower()',
    'response = "yes"  # auto-answered'
)
content = content.replace(
    'remove_response = input("Do you want to remove these records and proceed? (yes/no): ").strip().lower()',
    'remove_response = "no"  # auto-answered'
)
content = content.replace('"total_sswt_tax_deducted"', '"total_swt_tax_deducted"')

tmp_dir = Path(tempfile.gettempdir())
script2_patched = tmp_dir / '2_swt_validation_patched.py'
with open(script2_patched, 'w', encoding='utf-8') as f:
    f.write(content)
print('Patched validation script written to temp dir')

print(f'Running: 2_swt_validation_patched.py ...')
ok = run_script(script2_patched)
print(f'Step 2 finished, success={ok}')

# ── PRINT COLUMNS AFTER STEP 2 ───────────────────────────────────────────────
out2 = OUTPUT_DIR / 'swt_data_after_taxpayer_name_mapping.parquet'
if not out2.exists():
    out2 = OUTPUT_DIR / 'swt_cleaned_data.parquet'   # fallback
df_step2 = pd.read_parquet(out2)
print(f'\n=== COLUMNS AFTER STEP 2 — Validation & Cleaning ({len(df_step2.columns)}) ===')
print(df_step2.columns.tolist())

## STEP 3 — Feature Engineering & Rule Checking
Script: `3_swt_feature_engineering.py`  →  saves `final_output/swt_data_after_rule_checking.parquet`

In [ ]:
script3 = Path(SWT_DIR) / '3_swt_feature_engineering.py'
print(f'Running: {script3.name} ...')
ok = run_script(script3)
print(f'Step 3 finished, success={ok}')

# ── PRINT COLUMNS AFTER STEP 3 ───────────────────────────────────────────────
out3 = OUTPUT_DIR / 'swt_data_after_rule_checking.parquet'
df_step3 = pd.read_parquet(out3)
print(f'\n=== COLUMNS AFTER STEP 3 — Feature Engineering ({len(df_step3.columns)}) ===')
print(df_step3.columns.tolist())

## STEP 5 — Fraud Justification
Script: `5_swt_justification.py`  →  saves `final_output/swt_fraud_justification.parquet/.csv`

In [ ]:
# Patch justification script for PGK currency
script5_orig = Path(SWT_DIR) / '5_swt_justification.py'
with open(script5_orig, 'r', encoding='utf-8') as f:
    content5 = f.read()
content5 = content5.replace('(Rs.', '(PGK').replace('Rs.', 'PGK')
script5_patched = tmp_dir / '5_swt_justification_patched.py'
with open(script5_patched, 'w', encoding='utf-8') as f:
    f.write(content5)
print('Patched justification script written to temp dir')

print('Running: 5_swt_justification_patched.py ...')
ok = run_script(script5_patched)
print(f'Step 5 finished, success={ok}')

# ── PRINT COLUMNS AFTER STEP 5 ───────────────────────────────────────────────
out5_parq = OUTPUT_DIR / 'swt_fraud_justification.parquet'
out5_csv  = OUTPUT_DIR / 'swt_fraud_justification.csv'

if out5_parq.exists():
    df_step5 = pd.read_parquet(out5_parq)
    print('Reading from: swt_fraud_justification.parquet')
elif out5_csv.exists():
    df_step5 = pd.read_csv(out5_csv)
    print('Reading from: swt_fraud_justification.csv')
else:
    df_step5 = None
    print('Justification output not found.')

if df_step5 is not None:
    print(f'\n=== COLUMNS AFTER STEP 5 — Fraud Justification ({len(df_step5.columns)}) ===')
    print(df_step5.columns.tolist())

## Column Summary Across All Steps

In [ ]:
print('=== COLUMN COUNT SUMMARY ===')
print(f'  Raw Input       : {len(raw_df.columns)} columns')
print(f'  After Step 1    : {len(df_step1.columns)} columns')
print(f'  After Step 2    : {len(df_step2.columns)} columns')
print(f'  After Step 3    : {len(df_step3.columns)} columns')
if df_step5 is not None:
    print(f'  After Step 5    : {len(df_step5.columns)} columns')

print('\n=== NEW COLUMNS ADDED AT STEP 3 (rule/feature flags) ===')
new_cols_step3 = [c for c in df_step3.columns if c not in df_step2.columns]
print(new_cols_step3)